# 02 · Providers and endpoints

What the SDK actually does, and why one client works across vendors.
Raw HTTP vs SDK, OpenAI-compatible endpoints (Gemini, Ollama), and local reasoning models.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # repo root, for `toolkit`

In [ ]:
import os
import re

import requests
from dotenv import load_dotenv

load_dotenv(override=True)
openai_key = os.getenv("OPENAI_API_KEY")

## Raw HTTP

In [ ]:
response = requests.post(
    "https://api.openai.com/v1/chat/completions",
    headers={"Authorization": f"Bearer {openai_key}"},
    json={
        "model": "gpt-6-luna",
        "reasoning_effort": "none",
        "messages": [{"role": "user", "content": "One surprising fact about octopuses."}],
    },
    timeout=60,
)
body = response.json()
print(body["choices"][0]["message"]["content"])
print("\nusage:", body.get("usage"))

## SDK - same request, typed response objects

In [ ]:
from openai import OpenAI

reply = OpenAI().chat.completions.create(
    model="gpt-6-luna",
    reasoning_effort="none",
    messages=[{"role": "user", "content": "One surprising fact about octopuses."}],
)
reply.choices[0].message.content

## OpenAI-compatible endpoints

Most vendors expose the same Chat Completions contract, so switching provider is configuration:
`base_url`, `api_key`, model name. `toolkit.llm.PROVIDERS` holds those per provider.

In [ ]:
from toolkit import PROVIDERS, ask

for name, provider in PROVIDERS.items():
    print(f"{name:7} model={provider.default_model:22} base_url={provider.base_url or 'default'}")

In [ ]:
question = "In one sentence: what is a transformer in machine learning?"

for name in PROVIDERS:
    try:
        print(f"{name}: {ask(question, provider=name)}\n")
    except Exception as exc:
        print(f"{name}: skipped ({exc})\n")

## Local models (Ollama)

Zero marginal cost and data stays on-device, at the price of much lower capability.

In [ ]:
requests.get("http://localhost:11434").text

In [ ]:
!ollama list

### Reasoning output

`deepseek-r1:1.5b` (R1 distilled into Qwen) emits its chain of thought in `<think>` tags
before the answer; it needs stripping before display.

In [ ]:
raw = ask("Is 221 a prime number?", provider="ollama", model="deepseek-r1:1.5b")

thinking = re.search(r"<think>(.*?)</think>", raw, re.DOTALL)
final = re.sub(r"<think>.*?</think>", "", raw, flags=re.DOTALL).strip()

print("THINKING:\n", thinking.group(1).strip()[:800] if thinking else "(none)")
print("\nANSWER:\n", final)